### 之前三个模型统一训练参数训练的结果不是很好，attention模型出现了很多漏检的情况，resunet也有很多漏检的情况，所以针对三个模型进行训练调参然后重新训练看看结果有没有改善
具体怎么调参呢？
（1）大体的原则是先使用lr_finder设置一个初始的lr，他的原理是：
从非常小的 lr 开始
比如 1e-7
        ↓
每几个 batch 不断增加
        ↓
观察 training loss
        ↓
找到 loss 下降最快、
但还没有开始爆炸的位置
（2）然后使用ReduceLROnPlateau	Val 指标停滞时，就确定这个学习率
（3）Differential Learning Rate
但是 Attention 模型还有一个更专业的问题
ResNet18-U-Net和：Attention-ResNet18-U-Net

都用了 ImageNet pretrained encoder。
其实 Encoder 和 Decoder 不一定应该使用相同学习率。
因为：
Encoder
已经预训练过
→ 不希望更新太猛
Decoder / Attention
随机初始化或任务特异
→ 需要学得更快
$$
LR_{decoder}=3\times LR_{encoder}
$$

# 总结一下最后的调参路线
### Stage 1
确定一个合理 initial LR

ResNet:
1e-4 / 3e-4 / 1e-3

Attention:
1e-4 / 3e-4 / 1e-3

所有实验：
AdamW
+
ReduceLROnPlateau
+
Early Stopping

### Stage 2

固定 best initial LR
↓
比较 weight decay
0 / 1e-5 / 1e-4

### Stage 3

固定最佳配置
↓
Dice+BCE
vs
Tversky

# 规范的迁移学习优化链
LR Finder 找有效初始学习率 → Differential LR 区分预训练 Encoder 和新初始化 Decoder → ReduceLROnPlateau 根据 Val Dice 自动降学习率 → Early Stopping → 最后再和 Shared LR 做对照。

# 最后的改进模型
| 实验 | 模型 | Encoder | Decoder | LR策略 |
|---|---|---|---|---|
| R-S | ResNet18-U-Net | ImageNet pretrained | U-Net decoder | Shared LR |
| R-D | ResNet18-U-Net | ImageNet pretrained | U-Net decoder | **Differential LR** |
| A-S | Attention-ResNet18-U-Net | ImageNet pretrained | scSE decoder | Shared LR |
| A-D | Attention-ResNet18-U-Net | ImageNet pretrained | scSE decoder | **Differential LR** |

其中：
Shared LR
Encoder = Decoder

例如：
Encoder lr = 1e-4
Decoder lr = 1e-4

而 Differential LR：
预训练 Encoder
lr 较小

新任务 Decoder / Segmentation Head
lr 较大

比如最终可能是：
Encoder = 8e-5
Decoder = 2.4e-4